# NGS QC Data Analytics Pipeline

# NGS QC -  Data Extraction Pipeline

This notebook extracts quality control (QC) metrics from PDF reports and prepares a structured dataframe for downstream analysis and visualization.

## Steps:
1. Load PDF files
2. Extract relevant QC metrics
3. Clean and standardize data
4. Export dataframe for dashboard use

In [ ]:
!pip install OpenAI
!pip install pypdf
!pip install -q langchain langchain-community openai chromadb pypdf



# Using Langchain to Load Documents

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:

import os
from langchain_community.document_loaders import PyPDFLoader
from google.colab import files

# 1) Upload PDF files
uploaded = files.upload()  # pick one or more PDF reports
pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
if not pdf_paths:
    raise SystemExit("No PDFs uploaded.")

# 2) Load ALL pages from each PDF
all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load_and_split()   # load all pages (no truncation)
    all_docs.append({"path": path, "pages": pages})

# 3) Quick preview of the first 600 characters per document
for doc in all_docs:
    print(f"\n=== {doc['path']} — {len(doc['pages'])} pages ===")
    if doc["pages"]:
        print(doc["pages"][0].page_content[:600])

# 4) Flatten into one list for embedding or text search
all_pages = []
for doc in all_docs:
    for i, page in enumerate(doc["pages"], start=1):
        page.metadata["source_file"] = doc["path"]
        page.metadata["page_number"] = i
        all_pages.append(page)

print(f"\nLoaded {len(all_docs)} PDFs with {len(all_pages)} total pages.")


# Parsing the data

In [ ]:
import re
import pandas as pd
from dateutil import parser as dtparser
from pathlib import Path


# ---------------------------
# Helpers
# ---------------------------

def _to_number(val):
    if val is None:
        return None
    v = str(val).replace(",", "").strip()
    v = re.sub(r"\s+", " ", v)

    if re.search(r"[A-Za-z]", v):
        return v

    try:
        if re.fullmatch(r"-?\d+", v):
            return int(v)
        if re.fullmatch(r"-?\d+\.\d+", v):
            return float(v)
    except Exception:
        pass

    return v


def _parse_datetime_fuzzy(raw):
    if not raw:
        return None

    s = str(raw).strip()
    s = s.replace("Sept.", "Sep.").replace("Sept", "Sep")
    s = s.replace("a.m.", "AM").replace("p.m.", "PM")
    s = re.sub(r"\s+", " ", s)

    try:
        return pd.to_datetime(dtparser.parse(s, fuzzy=True))
    except Exception:
        return None


def extract_analysis_run_date(text):
    m = re.search(r"Analysis\s+Details", text, re.IGNORECASE)
    if not m:
        return (None, None)

    window = text[m.start(): m.start() + 12000]

    m2 = re.search(r"Analysis\s*Date\s*[:\-]?\s*(.+?)(?:\n|$)", window, re.IGNORECASE)
    if not m2:
        m2 = re.search(r"Run\s*Date\s*[:\-]?\s*(.+?)(?:\n|$)", window, re.IGNORECASE)

    if not m2:
        return (None, None)

    raw = re.sub(r"\s+", " ", m2.group(1)).strip()
    return (raw, _parse_datetime_fuzzy(raw))


# ---------------------------
# Main parser
# ---------------------------

def parse_ngs_metrics(text, source_file):
    data = {"Source_File": source_file}

    patterns = {
        "Run_Name": r"Run Report for\s+(.+?)(?:\n|$)",
        "ISP_Loading_%": r"Key Signal\s*\n\s*([\d.]+)\s*%",
        "Instrument": r"Instrument\s+([\w\-]+)",
        "Chip_Barcode": r"Chip Barcode\s+(\w+)",
        "Chip_Lot_Number": r"Chip Lot Number\s+(\w+)",
        "Chip_Wafer": r"Chip Wafer\s+(\d+)",
    }

    for key, pattern in patterns.items():
        m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        data[key] = _to_number(m.group(1)) if m else None

    # --- Total Reads robust extraction ---
    tr_patterns = [
        r"Total\s+Reads\s*\n\s*([\d,]{4,})",
        r"\b([\d,]{4,})\s*\n\s*Total\s+Reads\b",
        r"Total\s+Reads\s*[:\-]?\s*([\d,]{4,})",
    ]

    for pat in tr_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            data["Total_Reads"] = _to_number(m.group(1))
            break

    # --- Date ---
    raw_date, parsed_date = extract_analysis_run_date(text)
    data["Run_Date_raw"] = raw_date
    data["Run_Date"] = parsed_date

    return data


# ---------------------------
# Pipeline execution
# ---------------------------

records = []

for doc in all_docs:
    full_text = "\n".join([p.page_content for p in doc["pages"]])
    records.append(parse_ngs_metrics(full_text, doc["path"]))

df_metrics = pd.DataFrame(records)

df_metrics["Run_Date"] = pd.to_datetime(df_metrics["Run_Date"], errors="coerce")
df_metrics = df_metrics.sort_values("Run_Date", na_position="last")

print("Extracted metrics:")
print(df_metrics.head())


# ---------------------------
# Save output
# ---------------------------

output_path = Path("data/ngs_run_metrics.csv")
output_path.parent.mkdir(exist_ok=True)

df_metrics.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

# Some verifications before final analysis

In [ ]:
df_metrics_sorted = df_metrics.sort_values("Run_Date")


In [ ]:
summary = (
    pd.DataFrame({
        "column": df_metrics.columns,
        "total_rows": len(df_metrics),
        "missing_count": df_metrics.isna().sum().values,
    })
)

summary["missing_pct"] = (
    summary["missing_count"] / summary["total_rows"] * 100
).round(1)

display(summary.sort_values("missing_pct", ascending=False))


# NEW ANALYSIS - Hampel / MAD z-score

In [ ]:
# ---------------------------
# Hampel/MAD QC: multi-metric export
# ---------------------------

import numpy as np
import pandas as pd
from pathlib import Path


def add_hampel_mad_columns_multi(
    df: pd.DataFrame,
    date_col: str = "Run_Date",
    value_cols=("Mean_Read_Length_bp",),
    group_cols=("Instrument",),
    window: int = 15,
    k: float = 3.5,
    min_periods: int = 8,
    drop_dupes: bool = True,
    outlier_na=pd.NA,
) -> pd.DataFrame:
    """Add rolling Hampel/MAD robust baseline, robust z-score, and outlier flags."""

    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    value_cols = list(value_cols)
    group_cols = list(group_cols) if group_cols else []

    for metric in value_cols:
        if metric not in d.columns:
            raise KeyError(f"Missing metric column: {metric}")
        d[metric] = pd.to_numeric(d[metric], errors="coerce")

    for group_col in group_cols:
        if group_col not in d.columns:
            raise KeyError(f"Missing group column: {group_col}")

    sort_cols = group_cols + [date_col] if group_cols else [date_col]
    d = d.sort_values(sort_cols)

    if drop_dupes and group_cols:
        d = d.drop_duplicates(subset=group_cols + [date_col], keep="last")

    def _compute(group_df: pd.DataFrame) -> pd.DataFrame:
        group_df = group_df.sort_values(date_col).copy()

        for metric in value_cols:
            rolling_median = group_df[metric].rolling(
                window=window,
                min_periods=min_periods
            ).median()

            rolling_mad = (
                (group_df[metric] - rolling_median)
                .abs()
                .rolling(window=window, min_periods=min_periods)
                .median()
            )

            robust_sigma = 1.4826 * rolling_mad
            robust_z = (group_df[metric] - rolling_median) / robust_sigma

            group_df[f"robust_median__{metric}"] = rolling_median
            group_df[f"mad__{metric}"] = rolling_mad
            group_df[f"robust_sigma__{metric}"] = robust_sigma
            group_df[f"robust_z__{metric}"] = robust_z
            group_df[f"H_outlier__{metric}"] = np.where(
                robust_z.isna(),
                outlier_na,
                robust_z.abs() > k
            )

        return group_df

    if group_cols:
        parts = []
        for keys, group_df in d.groupby(group_cols, sort=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            for group_col, value in zip(group_cols, keys):
                group_df[group_col] = value
            parts.append(_compute(group_df))

        return (
            pd.concat(parts, ignore_index=True)
            .sort_values(group_cols + [date_col])
            .reset_index(drop=True)
        )

    return _compute(d).reset_index(drop=True)


# ---------------------------
# Load metrics dataframe
# ---------------------------

input_path = Path("data/ngs_run_metrics.csv")
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

df_metrics = pd.read_csv(input_path)
df_metrics["Run_Date"] = pd.to_datetime(df_metrics["Run_Date"], errors="coerce")


# ---------------------------
# Define QC metrics
# ---------------------------

metrics_to_include = [
    "Total_Reads",
    "Usable_Reads_%",
    "Mean_Read_Length_bp",
    "Median_Read_Length_bp",
    "Mode_bp",
]

metrics_to_include = [m for m in metrics_to_include if m in df_metrics.columns]

for metric in metrics_to_include:
    df_metrics[metric] = pd.to_numeric(df_metrics[metric], errors="coerce")


# ---------------------------
# Run Hampel/MAD by grouping factor
# ---------------------------

grouping_options = [
    ("Instrument",),
    ("Chip_Lot_Number",),
    ("Cleaning_Solution_Lot",),
    ("ExT_Sequencing_Reagent_Lot",),
    ("ExT_Wash_Solution_Lot",),
]

for group_cols in grouping_options:
    if any(group_col not in df_metrics.columns for group_col in group_cols):
        print(f"Skipping {group_cols}: column not found.")
        continue

    df_hampel = add_hampel_mad_columns_multi(
        df=df_metrics,
        date_col="Run_Date",
        value_cols=tuple(metrics_to_include),
        group_cols=group_cols,
        window=15,
        k=3.5,
        min_periods=8,
        drop_dupes=False,
    )

    group_name = "__".join(group_cols)
    output_path = output_dir / f"ngs_qc_hampel_mad_results__by_{group_name}.csv"

    df_hampel.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

# Visualization Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Exclude non-informative or categorical fields
cols_to_exclude = ["Addressable_Wells", "Chip_Type"]

# Filter dataframe
df_corr = df_metrics.drop(columns=cols_to_exclude, errors="ignore")

# Compute correlation matrix (numeric columns only)
corr = df_corr.corr(numeric_only=True)

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax)

ax.set_title("Correlation Matrix of NGS Metrics (Filtered)")
plt.tight_layout()
plt.show()

# Statistical Analysis of differences between instruments


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu

# Keep valid values only
df_test = df_metrics[
    df_metrics["Instrument"].notna() &
    df_metrics["Total_Reads"].notna()
].copy()


In [ ]:
insts = df_test["Instrument"].unique()
assert len(insts) == 2, "This test is for exactly 2 instruments"

x = df_test[df_test["Instrument"] == insts[0]]["Total_Reads"]
y = df_test[df_test["Instrument"] == insts[1]]["Total_Reads"]

u_stat, p_value = mannwhitneyu(x, y, alternative="two-sided")

print(f"Mann–Whitney U test")
print(f"Instruments: {insts[0]} vs {insts[1]}")
print(f"U statistic = {u_stat:.2f}")
print(f"p-value = {p_value:.4g}")


| p-value       | Interpretation                             |
| ------------- | ------------------------------------------ |
| **p ≥ 0.05**  | No statistically significant difference    |
| **p < 0.05**  | Statistically significant difference       |
| **p < 0.01**  | Strong evidence of difference              |
| **p < 0.001** | Very strong evidence (systemic difference) |
